## Apriori & FP tree

In [ ]:
!pip install pip --upgrade

In [ ]:
!pip install mlxtend==0.18

In [3]:
import mlxtend
mlxtend.__version__

'0.18.0'

In [4]:
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [5]:
import pandas as pd
import os

## Dataset import 및 확인

In [6]:
from google.colab import drive
drive.mount('/content/drive')

basicpath = '/content/drive/MyDrive/'

Mounted at /content/drive


In [23]:
path = basicpath + 'Temp'
file = 'BreadBasket_DMS.csv'
data = pd.read_csv(os.path.join(path, file), index_col = None)

In [24]:
data.head(10)

,Date,Time,Transaction,Item
0,2016.10.30,9:58:11,1,Bread
1,2016.10.30,10:05:34,2,Scandinavian
2,2016.10.30,10:05:34,2,Scandinavian
3,2016.10.30,10:07:57,3,Hot chocolate
4,2016.10.30,10:07:57,3,Jam
5,2016.10.30,10:07:57,3,Cookies
6,2016.10.30,10:08:41,4,Muffin
7,2016.10.30,10:13:03,5,Coffee
8,2016.10.30,10:13:03,5,Pastry
9,2016.10.30,10:13:03,5,Bread


In [9]:
data.shape

(21293, 4)

## 과제1-1 Data Cleaning


위 데이터를 가공하여, transaction과 item이 column이 되는  
다음과 같은 형태의 새로운 DataFrame을 만들어 변수 data에 저장하시오.

 transaction | item
:-------------:|:------:
      0      |   7  
      0      |   14
      1      |   9  
      2      |   18
...  


In [25]:
# 1. 대문자 -> 소문자 변환 (str.lower() 활용)

#Transaction과 Item만 꺼내기
data = data[['Transaction', 'Item']]

#Item의 elements에 lower 적용
data['Item'] = data['Item'].str.lower()

In [26]:
# 2. None value 제거 (drop 활용)

#NaN 제거
data = data.dropna(subset=['Item'])

#None 값 제거
data = data[data['Item'] != 'none']

# String 값과 int값 mapping
data['Item'] = data['Item'].astype('category').cat.codes

### Data 간단 분석_item당 등장횟수

In [27]:
data

,Transaction,Item
0,1,11
1,2,74
2,2,74
3,3,48
4,3,49
...,...,...
21288,9682,23
21289,9682,83
21290,9683,23
21291,9683,65


In [28]:
len(data['Item'].unique())

94

In [29]:
top10_items = data['Item'].value_counts().head(10)
top10_items

,count
Item,
23,5471
11,3325
83,1435
15,1025
65,856
73,771
55,616
48,590
26,540


In [30]:
len(data['Transaction'].unique())

9465

## 과제 1-2 one-hot-encoding vector 만들기
위 data를 가공하여 one-hot-encoding 형태의 데이터를 만들고  
이를 hot_encoded_data 변수에 저장하시오  
이때 one-hot-encoding 형태의 column은 item, row는 transaction이어야 함  

In [32]:
#답변
#Transaction 당 item 종류들을 몇개 샀는지
hot_encoded_data = pd.crosstab(data['Transaction'], data['Item'])

#하나라도 샀다면 1,하나도 사지 않았다면 0로 만들기
hot_encoded_data = (hot_encoded_data > 0).astype(int)

print(hot_encoded_data)

Item         0   1   2   3   4   5   6   7   8   9   ...  84  85  86  87  88  \
Transaction                                          ...                       
1             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
2             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
3             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
4             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
5             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
...          ..  ..  ..  ..  ..  ..  ..  ..  ..  ..  ...  ..  ..  ..  ..  ..   
9680          0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
9681          0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   1   
9682          0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
9683          0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
9684          0   0   0   0   0   0   0 

In [ ]:
#아래의 코드를 설명하시오:
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

hot_encoded_data = hot_encoded_data.applymap(encode_units)

#위의 코드와 같은 역할을 하고 있음. hot_enocoded_data가 0이하라면 0으로 만들고 1이상이라면 1로 만들기
#갯수가 아닌 존재하는지, 하지 않는지로 변환

<ipython-input-58-54271e18feba>:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  hot_encoded_data = hot_encoded_data.applymap(encode_units)


## 과제 1-3
mlxtend의 apriori & fp-growth 라이브러리를 활용하여 위 데이터에서 frequent itemset을 추출하시오.  
이때, min_support는 0.02로 하시오.  
association_rules 라이브러리를 활용하여 추출한 frequent itemset에서 association rule을 만드시오.

In [33]:
from mlxtend import frequent_patterns
#답변 apriori

#frequent_itemsets 찾기
frequent_itemsets_apriori = apriori(hot_encoded_data, min_support=0.02, use_colnames=True)

In [34]:
#Frequent_itemsets로 association_rules 만들기
rules = association_rules(frequent_itemsets_apriori, metric='lift', min_threshold=1)

In [35]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,(65),(11),0.086107,0.327205,0.029160,0.338650,1.034977,0.000985,1.017305
1,(11),(65),0.327205,0.086107,0.029160,0.089119,1.034977,0.000985,1.003306
2,(15),(23),0.103856,0.478394,0.054728,0.526958,1.101515,0.005044,1.102664
3,(23),(15),0.478394,0.103856,0.054728,0.114399,1.101515,0.005044,1.011905
4,(83),(15),0.142631,0.103856,0.023772,0.166667,1.604781,0.008959,1.075372
5,(15),(83),0.103856,0.142631,0.023772,0.228891,1.604781,0.008959,1.111865
6,(26),(23),0.054411,0.478394,0.028209,0.518447,1.083723,0.002179,1.083174
7,(23),(26),0.478394,0.054411,0.028209,0.058966,1.083723,0.002179,1.004841
8,(48),(23),0.058320,0.478394,0.029583,0.507246,1.060311,0.001683,1.058553
9,(23),(48),0.478394,0.058320,0.029583,0.061837,1.060311,0.001683,1.003749


In [36]:
#답변

#frequent_itemsets 찾기
frequent_itemset_fp = fpgrowth(hot_encoded_data, min_support=0.02, use_colnames=True)

In [37]:
#Frequent_itemsets로 association_rules 만들기
rules = association_rules(frequent_itemset_fp, metric='lift', min_threshold=1)

In [38]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,(48),(23),0.058320,0.478394,0.029583,0.507246,1.060311,0.001683,1.058553
1,(23),(48),0.478394,0.058320,0.029583,0.061837,1.060311,0.001683,1.003749
2,(26),(23),0.054411,0.478394,0.028209,0.518447,1.083723,0.002179,1.083174
3,(23),(26),0.478394,0.054411,0.028209,0.058966,1.083723,0.002179,1.004841
4,(65),(23),0.086107,0.478394,0.047544,0.552147,1.154168,0.006351,1.164682
5,(23),(65),0.478394,0.086107,0.047544,0.099382,1.154168,0.006351,1.014740
6,(65),(11),0.086107,0.327205,0.029160,0.338650,1.034977,0.000985,1.017305
7,(11),(65),0.327205,0.086107,0.029160,0.089119,1.034977,0.000985,1.003306
8,(55),(23),0.061807,0.478394,0.035182,0.569231,1.189878,0.005614,1.210871
9,(23),(55),0.478394,0.061807,0.035182,0.073542,1.189878,0.005614,1.012667
